# Data Generation for Synthetic RCT Diagnostics

This notebook explains what the stored synthetic dataset looks like, how it is generated, and how to inspect its distributions.

The project now enforces a versioned persistence policy:

1. Look for `data/synthetic/dataset_v1/campaigns.parquet`.
2. If it exists, load it.
3. If it does not exist, generate it once and save it.

That means notebooks, scripts, and evaluations all read from the same disk-backed dataset by default.

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd

from rct_diagnosis_agent.data import DatasetConfig, load_or_generate_data

In [ ]:
config = DatasetConfig(count=25, seed=7)
experiments = load_or_generate_data(config)
config.path, len(experiments), experiments[0].dataset_id, experiments[0].random_seed

## What one stored record contains

Each row in the parquet dataset is one `ExperimentSummary` record. The generator keeps the original experiment summary fields and extends them with more realistic ad-measurement structure:

- campaign-level fields such as impressions, clicks, spend, conversions, average conversion value, and revenue
- experiment-level fields such as control users, treatment users, and group conversions
- structured latent factors
- hidden causal truth used for evaluation

The agent still receives the same style of experiment summary as before, but the underlying synthetic world is more realistic.

In [ ]:
experiments[0].model_dump()

## Converting the dataset to an analysis table

A compact DataFrame makes it easier to inspect the stored campaign and experiment summaries side by side.

In [ ]:
rows = []
for exp in experiments:
    rows.append(
        {
            "experiment_id": exp.experiment_id,
            "dataset_id": exp.dataset_id,
            "impressions": exp.impressions,
            "clicks": exp.clicks,
            "spend": exp.spend,
            "revenue": exp.revenue,
            "control_users": exp.control_users,
            "treatment_users": exp.treatment_users,
            "total_users": exp.control_size + exp.treatment_size,
            "control_conversion_rate": exp.control_conversion_rate,
            "treatment_conversion_rate": exp.treatment_conversion_rate,
            "observed_conversion_diff": exp.treatment_conversion_rate - exp.control_conversion_rate,
            "pre_period_gap": exp.treatment_pre_mean - exp.control_pre_mean,
            "true_baseline_conversion_rate": exp.true_baseline_conversion_rate,
            "true_treatment_effect": exp.true_treatment_effect,
            "true_incremental_conversions": exp.true_incremental_conversions,
            "true_iROAS": exp.true_iROAS,
            "noise_level": exp.latent_factors.noise_level,
            "has_segments": bool(exp.segment_summaries),
            "issue_count": len(exp.hidden_truth),
            "hidden_truth": list(exp.hidden_truth),
        }
    )

df = pd.DataFrame(rows)
df.head()

## Issue frequency in the stored dataset

The generator can inject SRM, pre-period imbalance, low power, outliers, and Simpson's paradox. This quick count shows how often those cases appear in the persisted benchmark.

In [ ]:
issue_counts = Counter(issue for exp in experiments for issue in exp.hidden_truth)
pd.Series(issue_counts).sort_values(ascending=False)

## Latent factors

The old free-form notes are still present for compatibility, but the more important hidden structure is now carried in `latent_factors`.

In [ ]:
pd.DataFrame(
    [
        {
            "pre_period_imbalance": exp.latent_factors.pre_period_imbalance,
            "outliers": exp.latent_factors.outliers,
            "treatment_heterogeneity": exp.latent_factors.treatment_heterogeneity,
            "noise_level": exp.latent_factors.noise_level,
        }
        for exp in experiments
    ]
).value_counts().reset_index(name="count")

## Distribution summaries

The table below gives a compact view of the range of campaign scale, spend, rates, and hidden causal quantities represented in the persisted dataset.

In [ ]:
df[[
    "impressions",
    "clicks",
    "spend",
    "revenue",
    "total_users",
    "observed_conversion_diff",
    "true_treatment_effect",
    "true_iROAS",
]].describe().T

## Campaign scale distribution

Impressions vary from tens of thousands to millions, which makes the synthetic campaigns feel more like ad-measurement inputs than generic toy experiments.

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df["impressions"], bins=10, color="#4C78A8", edgecolor="white")
plt.title("Distribution of campaign impressions")
plt.xlabel("Impressions")
plt.ylabel("Number of campaigns")
plt.show()

## Spend versus revenue

Spend and revenue are generated at the campaign level. Their relationship is useful context when later interpreting hidden iROAS and treatment lift.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(df["spend"], df["revenue"], color="#54A24B", alpha=0.7)
plt.title("Campaign spend vs revenue")
plt.xlabel("Spend")
plt.ylabel("Revenue")
plt.show()

## Observed lift and hidden treatment effect

The observed treatment-control conversion difference is noisy because conversions are generated from binomial processes. The hidden treatment effect is the cleaner causal quantity used for evaluation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df["observed_conversion_diff"], bins=10, color="#F58518", edgecolor="white")
axes[0].axvline(0.0, color="black", linestyle="--", linewidth=1)
axes[0].set_title("Observed conversion-rate difference")
axes[0].set_xlabel("Treatment - Control")

axes[1].hist(df["true_treatment_effect"], bins=10, color="#E45756", edgecolor="white")
axes[1].axvline(0.0, color="black", linestyle="--", linewidth=1)
axes[1].set_title("Hidden true treatment effect")
axes[1].set_xlabel("True effect")

plt.tight_layout()
plt.show()

## Pre-period linkage and post-period outcomes

The generator links pre and post periods explicitly. Post-period outcomes are built from pre-period levels, common trend, treatment effect, and noise.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(df["pre_period_gap"], df["observed_conversion_diff"], alpha=0.7, color="#72B7B2")
plt.axhline(0.0, color="black", linestyle="--", linewidth=1)
plt.axvline(0.0, color="black", linestyle="--", linewidth=1)
plt.title("Pre-period gap vs observed lift")
plt.xlabel("Treatment pre mean - Control pre mean")
plt.ylabel("Observed treatment-control conversion difference")
plt.show()

## Segment-bearing experiments

Simpson's paradox cases are represented through `segment_summaries`. These experiments keep segment-level structure in the stored dataset so the agent can reason about traffic-mix reversals.

In [ ]:
segment_examples = [exp for exp in experiments if exp.segment_summaries]
len(segment_examples), segment_examples[0].segment_summaries[0].model_dump() if segment_examples else None